# 01 · Exploratory Data Analysis
## Bitcoin Transaction Network — Illicit Activity Detection

> **Dataset:** Elliptic Bitcoin Dataset  
> **Nodes:** 203,769 transactions | **Edges:** 234,355 directed payment flows  
> **Labels:** 2.2% illicit · 20.6% licit · 77.1% unknown

---

### What this notebook does
Before building any model we need to deeply understand our data. This notebook answers five questions:
1. How severe is the class imbalance — and why does it matter for modelling?
2. Is illicit activity randomly distributed across time, or does it cluster in bursts?
3. Which of the 166 features best separate illicit from licit nodes?
4. Do illicit and licit nodes have different internal feature structures?
5. What does the transaction graph look like topologically?

Every finding here directly motivates a decision made in notebooks 02, 03, and 04.

### Output
- `df` — merged DataFrame with class labels (used by all subsequent notebooks)
- `G` — NetworkX directed graph (used by notebooks 02 and 04)
- `edges` — edge list DataFrame
- 5 saved chart images

## Setup — Download and Load Data

**Before running:** Download the Elliptic dataset from Kaggle and place the three CSV files in `data/elliptic_bitcoin_dataset/`.  
See `data/README.md` for instructions, or run the Kaggle download cells below.

In [ ]:
# Install Kaggle API
!pip install kaggle -q
print('✅ Kaggle installed')

In [ ]:
# Set up Kaggle credentials
# Replace YOUR_KAGGLE_USERNAME and YOUR_KAGGLE_KEY with your own credentials
# Get your API key from: kaggle.com → Settings → API → Create New Token
import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    f.write('{"username":"YOUR_KAGGLE_USERNAME","key":"YOUR_KAGGLE_KEY"}')
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle credentials set')

In [ ]:
# Download the Elliptic dataset from Kaggle
!kaggle datasets download -d ellipticco/elliptic-data-set
print('✅ Download complete')

In [ ]:
# Unzip the downloaded dataset
import zipfile
with zipfile.ZipFile('elliptic-data-set.zip', 'r') as z:
    z.extractall('data')
print('✅ Unzipped')

In [ ]:
# Confirm all three files are present
import os
for root, dirs, files in os.walk('data'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Load the three Elliptic dataset files into pandas DataFrames
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

FEATURES_PATH = 'data/elliptic_bitcoin_dataset/elliptic_txs_features.csv'
CLASSES_PATH  = 'data/elliptic_bitcoin_dataset/elliptic_txs_classes.csv'
EDGES_PATH    = 'data/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv'

feat_cols = ['txId', 'time_step'] + [f'f{i}' for i in range(1, 166)]
features  = pd.read_csv(FEATURES_PATH, header=None, names=feat_cols)
classes   = pd.read_csv(CLASSES_PATH)
edges     = pd.read_csv(EDGES_PATH)

print(f'Features : {features.shape}')
print(f'Classes  : {classes.shape}')
print(f'Edges    : {edges.shape}')
print('✅ All files loaded successfully')

In [ ]:
# Merge features and class labels into one DataFrame
# Convert class labels: 1=Illicit, 2=Licit, 0=Unknown
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

df = features.merge(classes, on='txId', how='left')
df['class'] = df['class'].replace({'unknown': 0, '1': 1, '2': 2}).astype(float)
df['class_label'] = df['class'].map({1.0: 'Illicit', 2.0: 'Licit', 0.0: 'Unknown'})

print('Class distribution:')
print(df['class_label'].value_counts())
print(f'\nTotal nodes: {len(df):,}')
labelled = df[df['class'] != 0]
ill_pct  = (labelled['class'] == 1).mean() * 100
print(f'Among labelled nodes: {ill_pct:.1f}% are illicit')
print('✅ Data merged successfully')

In [ ]:
# Set up dark theme and colour palette for all charts
plt.rcParams.update({
    'figure.facecolor' : '#0D1117',
    'axes.facecolor'   : '#161B22',
    'axes.edgecolor'   : '#30363D',
    'axes.labelcolor'  : '#C9D1D9',
    'xtick.color'      : '#8B949E',
    'ytick.color'      : '#8B949E',
    'text.color'       : '#C9D1D9',
    'grid.color'       : '#21262D',
    'grid.linestyle'   : '--',
    'figure.dpi'       : 130,
})
PALETTE = {'Illicit': '#FF4444', 'Licit': '#00C9A7', 'Unknown': '#8B949E'}
print('✅ Visualisation style set')

## 1. Class Distribution
**Finding:** Severe class imbalance — only 9.8% of labelled nodes are illicit.  
**Implication:** Accuracy is a misleading metric. We must use F1, AUC, and recall instead.

In [ ]:
# Plot class distribution — bar chart and pie chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Class Distribution in the Elliptic Dataset', fontsize=14, y=1.02)

counts  = df['class_label'].value_counts()
colours = [PALETTE[c] for c in counts.index]

axes[0].bar(counts.index, counts.values, color=colours, edgecolor='none', width=0.5)
for i, (lbl, val) in enumerate(counts.items()):
    axes[0].text(i, val + 1500, f'{val:,}', ha='center', fontsize=10)
axes[0].set_title('Absolute Counts')
axes[0].set_ylabel('Number of Transactions')

axes[1].pie(counts.values, labels=counts.index, colors=colours,
            autopct='%1.1f%%', startangle=140,
            textprops={'color': '#C9D1D9'},
            wedgeprops={'edgecolor': '#0D1117', 'linewidth': 2})
axes[1].set_title('Proportion')

plt.tight_layout()
plt.savefig('fig_class_distribution.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Class distribution chart saved')

## 2. Temporal Distribution
**Finding:** Illicit activity is NOT uniformly distributed — it spikes at specific time steps.  
**Implication:** Temporal features will be highly predictive. Train/test split must be temporal, not random.

In [ ]:
# Plot transaction activity across all 49 time steps
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle('Transaction Activity Across 49 Time Steps', fontsize=14)

ts = df.groupby(['time_step', 'class_label']).size().unstack(fill_value=0)

for col, colour in PALETTE.items():
    if col in ts.columns:
        axes[0].fill_between(ts.index, ts[col], alpha=0.7, color=colour, label=col)
axes[0].set_ylabel('Node Count')
axes[0].legend(loc='upper right')
axes[0].set_title('All Classes')

axes[1].bar(ts.index, ts['Illicit'], color='#FF4444', alpha=0.85, width=0.8)
peak_ts = ts['Illicit'].idxmax()
peak_v  = ts['Illicit'].max()
axes[1].annotate(
    f'Peak: {peak_v} nodes\n(step {peak_ts})',
    xy=(peak_ts, peak_v), xytext=(peak_ts + 2, peak_v * 1.05),
    arrowprops=dict(arrowstyle='->', color='#FF4444'),
    color='#FF4444', fontsize=9
)
axes[1].set_ylabel('Illicit Node Count')
axes[1].set_xlabel('Time Step')
axes[1].set_title('Illicit Nodes Only — Burst Pattern')

plt.tight_layout()
plt.savefig('fig_temporal_distribution.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Temporal distribution chart saved')

## 3. Feature Distributions
**Finding:** Illicit nodes cluster tightly around specific feature values. Licit nodes are more diverse.  
**Implication:** Several features have strong discriminative power for the classifier.

In [ ]:
# Find the top 6 most discriminative features and plot their distributions
labelled_df = df[df['class'] != 0].copy()
labelled_df['label'] = labelled_df['class'].map({1.0: 'Illicit', 2.0: 'Licit'})
local_feats = [f'f{i}' for i in range(1, 94)]

means = labelled_df.groupby('label')[local_feats].mean()
diff  = (means.loc['Illicit'] - means.loc['Licit']).abs().sort_values(ascending=False)
top6  = diff.head(6).index.tolist()

print('Top 6 most discriminative features:')
print(diff.head(6).round(4).to_string())

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions — Illicit vs Licit (Top 6 Most Discriminative)',
             fontsize=14)

for ax, feat in zip(axes.flat, top6):
    for lbl, colour in [('Illicit', '#FF4444'), ('Licit', '#00C9A7')]:
        vals = labelled_df[labelled_df['label'] == lbl][feat].dropna()
        p1, p99 = vals.quantile(0.01), vals.quantile(0.99)
        vals = vals.clip(p1, p99)
        ax.hist(vals, bins=40, alpha=0.6, color=colour, label=lbl, density=True)
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_feature_distributions.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Feature distribution chart saved')

## 4. Correlation Heatmap
**Finding:** Illicit node features are highly correlated with each other (deep red). Licit features are diverse.  
**Implication:** Fraud follows organised, stereotyped patterns — consistent with coordinated money laundering.

In [ ]:
# Correlation heatmap comparing feature structure inside illicit vs licit nodes
top15 = diff.head(15).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Feature Correlation Structure — Illicit vs Licit (Top 15 Features)',
             fontsize=13)

for ax, lbl, cmap in zip(axes, ['Illicit', 'Licit'], ['Reds', 'Greens']):
    subset = labelled_df[labelled_df['label'] == lbl][top15]
    corr   = subset.corr()
    sns.heatmap(corr, ax=ax, cmap=cmap, center=0,
                linewidths=0.4, linecolor='#0D1117',
                xticklabels=True, yticklabels=True,
                annot=False, cbar_kws={'shrink': 0.8})
    ax.set_title(lbl, color=PALETTE[lbl], fontsize=12)
    ax.tick_params(axis='both', labelsize=8)

plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Correlation heatmap saved')

## 5. Graph Topology
**Finding:** Power-law degree distribution (scale-free network). Max in-degree 284, max out-degree 472.  
**Implication:** A few hub nodes dominate the network — these are the highest structural risk nodes.

In [ ]:
# Build the directed transaction graph and plot degree distributions
print('Building graph — this may take ~30 seconds...')

G = nx.from_pandas_edgelist(
    edges, source='txId1', target='txId2',
    create_using=nx.DiGraph()
)

label_map = df.set_index('txId')['class_label'].to_dict()
nx.set_node_attributes(G, label_map, 'class_label')

in_deg  = dict(G.in_degree())
out_deg = dict(G.out_degree())

df_deg = df[['txId', 'class_label']].copy()
df_deg['in_degree']  = df_deg['txId'].map(in_deg).fillna(0).astype(int)
df_deg['out_degree'] = df_deg['txId'].map(out_deg).fillna(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Degree Distribution by Class', fontsize=13)

for ax, col, title in zip(axes, ['in_degree', 'out_degree'], ['In-Degree', 'Out-Degree']):
    for cls, colour in PALETTE.items():
        vals = df_deg[df_deg['class_label'] == cls][col]
        vals = vals[vals > 0]
        if len(vals) == 0: continue
        counts_v = vals.value_counts().sort_index()
        ax.loglog(counts_v.index, counts_v.values, '.',
                  color=colour, alpha=0.6, markersize=4, label=cls)
    ax.set_title(title)
    ax.set_xlabel('Degree (log)')
    ax.set_ylabel('Count (log)')
    ax.legend()

plt.tight_layout()
plt.savefig('fig_degree_distribution.png', bbox_inches='tight', dpi=130)
plt.show()

wcc = list(nx.weakly_connected_components(G))
print(f'\nGraph summary:')
print(f'  Nodes         : {G.number_of_nodes():,}')
print(f'  Edges         : {G.number_of_edges():,}')
print(f'  Avg in-degree : {np.mean(list(in_deg.values())):.2f}')
print(f'  Avg out-degree: {np.mean(list(out_deg.values())):.2f}')
print(f'  Max in-degree : {max(in_deg.values()):,}')
print(f'  Max out-degree: {max(out_deg.values()):,}')
print(f'  WCC count     : {len(wcc):,}')
print(f'  Largest WCC   : {max(len(c) for c in wcc):,} nodes')
print('\n✅ Graph topology analysis complete')

## EDA Summary

| Finding | Implication |
|---------|-------------|
| 77% unknown nodes | GNN needed to fill the labelling gap |
| 9.8% illicit among labelled | Class weighting needed in training |
| Illicit activity bursts at step 32 | Temporal features matter |
| Illicit features narrow and stereotyped | Strong discriminative signal exists |
| Illicit features highly correlated | Fraud follows organised patterns |
| Power-law degree distribution | Scale-free network — hubs are highest risk |
| Max in-degree 284, out-degree 472 | Extreme hubs warrant immediate investigation |

**Next:** Run `02_Feature_Engineering.ipynb`